In [1]:
import base64
import pandas as pd
import os
import time
import openai
from PIL import Image
import io
import csv
from dotenv import load_dotenv

# Load from .env
load_dotenv()

# Access the key
openai_api_key = os.getenv("OPENAI_API_KEY")

# Setup OpenAI client (new in v1+)
client = openai.OpenAI(api_key=openai_api_key)


In [2]:

prompt = """
    DO NOT SUMMARIZE. ONLY OUTPUT RAW TAGGED BLOCKS.
    Generate random QPE circuit with depth 8 and number of qubits 6 (Qiskit qasm), using logical gate and not basic circuit (little bit complex).
Supposed that the output is image, you need to make chain of thought on how to reverse the image into thinking process so it can summarize the details, the structures, and classify number of qubits along with registers, also how to classify it as grover/qpe/qml etc
Make sure OPENQASM run in Qiskit and no error.
This is the example format:

<think>  
1. Based on the image, consist of pattern .. there is CCNOT and X gate as oracle then it is classified as Grover (as detail as possible) (Might be different for QPE, QML etc) 
(You may add more steps here as detail as possible)

2. Number of qubits, how many quantum classical registers, is there measurement, reverse engineering from image to thinking process, how to classify number of qubits, registers, and measurement

3. Composition - Logical GATE (reverse OCR):  
[Q0] --- [X] --- [H] --- [Cnot1]  
[Q1] --- [X] --- [H] --- [Cnot1]  

(as detail as possible, you may add more here)
</think>  

<OPENQASM code> 
OPENQASM 2.0;
include "qelib1.inc";
 ...  
 ...  
</OPENQASM code>  
"""


response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a quantum optics assistant."},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    # {
                    #     "type": "image_url",
                    #     "image_url": {
                    #         "url": f"data:image/png;base64,{base64_image}",
                    #         "detail": "high"
                    #     }
                    # }
                ]
            }
        ],
        max_tokens=2000
    )


In [3]:

print(response.choices[0].message.content)

<think>  
1. Upon examining the image representing the quantum circuit, I note that the presence of a Hadamard gate, controlled rotations, and inverse Quantum Fourier Transform (iQFT) suggests that this circuit is related to the Quantum Phase Estimation (QPE) algorithm. The QPE algorithm typically involves preparing a superposition via Hadamard gates on the auxiliary qubits, applying the controlled unitary operations, and completing the process with an iQFT to discern the phase information. 

2. For determining the number of qubits and registers: A close examination of the circuit shows six quantum wires, corresponding to six qubits, with possible additional classical lines for measurement outcomes indicative of a quantum-classical hybrid architecture. The circuit's final section includes measurement gates that transduce quantum information into classical bit strings. Identifying registers involves recognizing the delineation between quantum operations and measurements, noting which qu

In [1]:
from qiskit import QuantumCircuit
from qiskit.visualization import circuit_drawer
 
# 1. Your OpenQASM code
qasm_code = """
OPENQASM 2.0;
include "qelib1.inc";

qreg q[7];
creg c[1];

// Initialization
h q[0];
h q[1];
h q[2];
h q[3];
h q[4];
h q[5];
h q[6];

// Controlled-U operations
rz(pi/2) q[6];
cx q[5], q[6];
rz(pi/4) q[6];
cx q[4], q[6];
rz(pi/8) q[6];
cx q[3], q[6];
rz(pi/16) q[6];
cx q[2], q[6];
rz(pi/32) q[6];
cx q[1], q[6];
rz(pi/64) q[6];
cx q[0], q[6];

// Measurement
measure q[6] -> c[0];
"""
 
# 2. Convert OpenQASM to QuantumCircuit
qc = QuantumCircuit.from_qasm_str(qasm_code)
 
# 3. Draw and save the circuit image
fig = circuit_drawer(qc, output='mpl')  # matplotlib figure
fig.savefig("circuit_output2.png")
 